# Colab 실행 안내

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glasslego/2026-ml-study/blob/main/notebooks/DLFS_code/03_dlfs/Code_pytorch.ipynb)

이 노트북은 같은 디렉토리의 `Code.ipynb` (밑바닥부터 numpy 로 직접 구현한 신경망) 와
**같은 추상화/같은 흐름**으로 동일한 회귀 문제를 **PyTorch** 를 사용해 다시 푼다.

흐름:

1. 환경 준비 (Colab/Local 공통)
2. Operation 대응 — PyTorch tensor 와 autograd
3. Layer 대응 — `nn.Module`, `nn.Linear`
4. Loss 대응 — `nn.MSELoss`
5. NeuralNetwork 대응 — `nn.Sequential` / 사용자 `nn.Module`
6. Optimizer 대응 — `torch.optim.SGD`
7. Trainer 대응 — `DataLoader` + 학습 루프
8. 데이터 로드 / 표준화 / 분할 (Boston 은 deprecated → California Housing)
9. 3가지 모델 비교 (선형회귀 / 1은닉층 / 2은닉층)

> 같은 연산을 책의 numpy 코드, PyTorch, TensorFlow 셋이 어떻게 표현하는지 비교하면서 보면
> 프레임워크가 무엇을 "자동" 으로 해 주는지 명확히 보인다.


In [ ]:
# 공통 실행 환경 준비
# 이 셀은 Colab과 로컬 Jupyter 사이의 환경 차이를 흡수한다.
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/glasslego/2026-ml-study.git'
REPO_NAME = '2026-ml-study'
DLFS_SUBDIR = 'notebooks/DLFS_code'

if 'google.colab' in sys.modules:
    _bootstrap_clone_root = Path('/content') / REPO_NAME
    if not _bootstrap_clone_root.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(_bootstrap_clone_root)], check=True)
    _bootstrap_repo_root = _bootstrap_clone_root / DLFS_SUBDIR
else:
    _bootstrap_repo_root = None
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / 'README.md').exists() and (candidate / 'lincoln').exists():
            _bootstrap_repo_root = candidate
            break
    if _bootstrap_repo_root is None:
        raise FileNotFoundError('DLFS_code 저장소 루트를 찾지 못했습니다.')

if str(_bootstrap_repo_root) not in sys.path:
    sys.path.insert(0, str(_bootstrap_repo_root))

from notebook_env import prepare_notebook_environment

REPO_ROOT = prepare_notebook_environment(
    notebook_dir='03_dlfs',
    needs_lincoln=False,
    ensure_mnist=False,
)


In [ ]:
# Colab 에는 PyTorch 가 기본 설치되어 있지만, 로컬에 없을 수도 있으므로 안전하게 보강한다.
try:
    import torch  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch'], check=True)
    import torch  # noqa: F401


In [ ]:
# 핵심 import.
# - torch          : tensor 와 autograd
# - torch.nn       : Layer / Loss 모듈 (책의 Layer / Loss 추상화에 대응)
# - torch.optim    : Optimizer (책의 Optimizer 추상화에 대응)
# - torch.utils.data : DataLoader (책의 generate_batches 에 대응)
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from typing import List, Tuple

# 재현성: 책의 seed=20190501 과 동일한 의도.
SEED = 20190501
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"torch version: {torch.__version__}")


# 1. Operation 대응 — Tensor 와 autograd

책의 `Operation` 추상화는 forward 와 backward 를 직접 구현하는 것이 핵심이었다.
PyTorch 에서는 **tensor 에 `requires_grad=True` 만 주면 forward 그래프가 자동으로 기록되고**,
`loss.backward()` 한 줄이 chain rule 을 알아서 적용한다.

아래 셀은 같은 "행렬곱 + 편향 + sigmoid" 흐름을 numpy 와 PyTorch 가 각각 어떻게 표현하는지 비교한다.


In [ ]:
# numpy 와 PyTorch 의 forward/backward 비교 (작은 예시)
rng = np.random.default_rng(0)
X_np = rng.standard_normal((4, 3)).astype(np.float32)
W_np = rng.standard_normal((3, 2)).astype(np.float32)
b_np = rng.standard_normal((1, 2)).astype(np.float32)

# (a) 책의 방식 — forward / backward 를 손으로 구현해야 한다.
def forward_numpy(X, W, b):
    Y = X @ W + b
    S = 1.0 / (1.0 + np.exp(-Y))
    L = float(S.sum())  # 가장 단순한 손실
    return Y, S, L

Y_np, S_np, L_np = forward_numpy(X_np, W_np, b_np)
print("numpy loss:", L_np)

# (b) PyTorch — requires_grad=True 만 주면 backward 가 자동.
X = torch.tensor(X_np)
W = torch.tensor(W_np, requires_grad=True)
b = torch.tensor(b_np, requires_grad=True)

S = torch.sigmoid(X @ W + b)
L = S.sum()
L.backward()  # autograd 가 chain rule 자동 적용
print("torch loss:", L.item())
print("dL/dW shape:", W.grad.shape, "  dL/db shape:", b.grad.shape)


# 2. Layer 대응 — `nn.Module` 과 `nn.Linear`

책의 `Dense` 층은 `[WeightMultiply -> BiasAdd -> activation]` 세 Operation 의 합성이었다.
PyTorch 에서는 이 합성을 `nn.Linear`(가중치 + 편향) + 활성화 함수 한 줄로 끝낸다.

아래 `DenseBlock` 은 책의 `Dense` 와 **이름까지 같은 의미**로 만든 얇은 래퍼다.
직접 만들어 보면 `nn.Sequential([nn.Linear, activation])` 와 정확히 같다는 것을 알 수 있다.


In [ ]:
class DenseBlock(nn.Module):
    """책의 Dense 층에 대응하는 PyTorch nn.Module.

    내부적으로:
        - nn.Linear        : WeightMultiply + BiasAdd 를 한 번에 (in_features -> out_features)
        - self.activation  : Sigmoid / Identity 등 (책의 Sigmoid / Linear 와 같은 역할)

    파라미터(W, b)와 그 기울기는 nn.Module 이 자동으로 관리한다.
    """

    def __init__(self, in_features: int, out_features: int, activation: nn.Module = nn.Sigmoid()):
        super().__init__()
        # nn.Linear 는 W (out, in) 와 b (out,) 를 자동으로 만들어 준다 (Kaiming uniform 초기화).
        self.linear = nn.Linear(in_features, out_features)
        self.activation = activation

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # forward 만 적어 두면 backward 는 autograd 가 알아서 만들어 준다.
        return self.activation(self.linear(x))


# 3. Loss 대응 — `nn.MSELoss`

책의 `MeanSquaredError` 는 `(1/N) * sum((y_hat - y)^2)` 와 그 기울기를 직접 구현했다.
PyTorch 의 `nn.MSELoss(reduction='mean')` 가 같은 손실을 제공하고, 기울기는 autograd 가 처리한다.


In [ ]:
loss_fn = nn.MSELoss(reduction='mean')

# 작은 예시로 책의 공식 dL/dy_hat = 2(y_hat - y)/N 가 그대로 나오는지 확인한다.
y_hat = torch.tensor([[1.0], [2.0], [3.0]], requires_grad=True)
y     = torch.tensor([[1.0], [1.0], [1.0]])
loss = loss_fn(y_hat, y)
loss.backward()
print("loss:", loss.item())                  # mean(((1-1)^2, (2-1)^2, (3-1)^2)) = (0+1+4)/3 = 5/3
print("dL/dy_hat:", y_hat.grad.flatten())    # 2*(y_hat - y)/N = [0, 2/3, 4/3]


# 4. NeuralNetwork 대응 — `nn.Sequential` / 사용자 `nn.Module`

책의 `NeuralNetwork` 는 `Layer` 의 리스트를 들고 forward / backward 를 직렬로 호출했다.
PyTorch 에서는 동일한 의미를 `nn.Sequential` 로 한 줄에 표현할 수 있다.

아래 `make_network()` 헬퍼는 책의 `NeuralNetwork(layers=[Dense(...), Dense(...), ...])` 와
역할이 같다.


In [ ]:
def make_network(layer_sizes: List[int], activations: List[nn.Module]) -> nn.Module:
    """책의 NeuralNetwork(layers=[Dense(...), ...]) 와 같은 역할.

    Args:
        layer_sizes: [입력차원, hidden1, hidden2, ..., 출력차원]
        activations: 각 DenseBlock 에 들어갈 활성화 모듈 리스트 (length = len(layer_sizes) - 1)
    """
    assert len(activations) == len(layer_sizes) - 1, \
        "activation 개수는 (layer_sizes - 1) 과 같아야 함"

    blocks: List[nn.Module] = []
    for i, act in enumerate(activations):
        blocks.append(DenseBlock(layer_sizes[i], layer_sizes[i + 1], activation=act))
    return nn.Sequential(*blocks)


# 5. Optimizer 대응 — `torch.optim.SGD`

책의 `SGD.step()` 은 `param -= lr * grad` 를 직접 수행했다.
PyTorch 의 `torch.optim.SGD(model.parameters(), lr=0.01)` 가 동일한 일을 한다.

차이점은 **PyTorch 는 매 step 직전 `optimizer.zero_grad()` 가 필요하다**는 것이다.
PyTorch 의 grad 는 누적(accumulating)되기 때문이다.


# 6. Trainer 대응 — `DataLoader` + 학습 루프

책의 `Trainer.fit()` 은:
1. epoch 마다 데이터를 셔플
2. 배치 단위로 forward / loss / backward / step
3. eval_every 마다 검증 셋에서 손실 측정 — 더 좋아졌으면 갱신, 나빠졌으면 직전 백업으로 복원하고 종료 (early stopping)

PyTorch 에서는 셔플과 배치 분할을 `DataLoader(shuffle=True, batch_size=...)` 가 대신해 준다.
나머지 학습 루프와 early stopping 은 우리가 직접 짜서 책과 1:1 로 매칭되게 한다.


In [ ]:
import copy

class TorchTrainer:
    """책의 Trainer 와 같은 인터페이스를 PyTorch 위에 구현한 미니 트레이너.

    책과 같은 부분:
        - eval_every 마다 검증 손실 측정 후 best 갱신 / 복원 / early stop
        - 같은 seed 로 실행하면 비교 가능한 결과
    PyTorch 가 대신해 주는 부분:
        - 미니배치 셔플 (DataLoader)
        - 파라미터 갱신 자체 (optimizer.step)
        - chain rule (loss.backward)
    """

    def __init__(self, net: nn.Module, optimizer: optim.Optimizer, loss_fn: nn.Module):
        self.net = net
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.best_loss = float('inf')

    def fit(self,
            X_train: torch.Tensor, y_train: torch.Tensor,
            X_test: torch.Tensor,  y_test: torch.Tensor,
            epochs: int = 50,
            eval_every: int = 10,
            batch_size: int = 32,
            seed: int = SEED) -> None:

        # 재현성 보장
        torch.manual_seed(seed)

        train_loader = DataLoader(
            TensorDataset(X_train, y_train),
            batch_size=batch_size,
            shuffle=True,   # 책의 permute_data 에 대응
        )

        self.best_loss = float('inf')

        for e in range(epochs):
            # eval 직전 모델/옵티마이저 백업 (early stopping 시 복원용)
            if (e + 1) % eval_every == 0:
                last_model_state = copy.deepcopy(self.net.state_dict())

            # --- 한 epoch 학습 ---
            self.net.train()
            for X_batch, y_batch in train_loader:
                self.optimizer.zero_grad()        # PyTorch 의 grad 는 누적되므로 매 step 초기화 필수
                preds = self.net(X_batch)
                loss = self.loss_fn(preds, y_batch)
                loss.backward()                   # autograd 가 chain rule 자동 처리
                self.optimizer.step()             # param -= lr * grad

            # --- 검증 손실 측정 + early stopping ---
            if (e + 1) % eval_every == 0:
                self.net.eval()
                with torch.no_grad():
                    test_preds = self.net(X_test)
                    val_loss = self.loss_fn(test_preds, y_test).item()

                if val_loss < self.best_loss:
                    print(f"{e + 1} 에폭에서 검증 데이터에 대한 손실값: {val_loss:.3f}")
                    self.best_loss = val_loss
                else:
                    print(
                        f"{e + 1} 에폭에서 손실값이 증가했다. 마지막으로 측정한 손실값은 "
                        f"{e + 1 - eval_every} 에폭까지 학습된 모델에서 계산된 {self.best_loss:.3f} 이다."
                    )
                    # 백업본으로 복원하고 학습 종료
                    self.net.load_state_dict(last_model_state)
                    break


# 7. 평가 함수

책의 `mae`, `rmse`, `eval_regression_model` 과 같은 의미의 헬퍼.


In [ ]:
def mae(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    return torch.mean(torch.abs(y_true - y_pred)).item()


def rmse(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    return torch.sqrt(torch.mean((y_true - y_pred) ** 2)).item()


def eval_regression_model(model: nn.Module, X_test: torch.Tensor, y_test: torch.Tensor) -> None:
    model.eval()
    with torch.no_grad():
        preds = model(X_test).reshape(-1, 1)
    print("평균절대오차: {:.2f}".format(mae(y_test, preds)))
    print()
    print("제곱근 평균제곱오차 {:.2f}".format(rmse(y_test, preds)))


# 8. 데이터 로드 — Boston 대신 California Housing

책에서는 `sklearn.datasets.load_boston` 을 사용했지만 scikit-learn 1.2 부터 윤리적 이유로 제거되었다.
같은 회귀 문제 형태(여러 수치 특성 -> 연속 target) 인 **California Housing** 데이터셋으로 대체한다.

- 특성 8개 (책의 13개와 다름)
- target: median house value (단위: 십만 달러)
- 절차는 동일: 표준화 -> train/test split -> (N, 1) reshape


In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

cali = fetch_california_housing()
data = cali.data            # shape: (20640, 8)
target = cali.target        # shape: (20640,)
features = cali.feature_names

# 표준화: sigmoid 포화 방지를 위해 책과 동일하게 평균 0 / 분산 1 로 맞춘다.
scaler = StandardScaler()
data = scaler.fit_transform(data).astype(np.float32)
target = target.astype(np.float32)

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    data, target, test_size=0.3, random_state=80718
)

# 책과 동일하게 target 을 (N, 1) 로 만든다.
y_train_np = y_train_np.reshape(-1, 1)
y_test_np  = y_test_np.reshape(-1, 1)

# numpy -> torch tensor 변환.
X_train = torch.from_numpy(X_train_np)
X_test  = torch.from_numpy(X_test_np)
y_train = torch.from_numpy(y_train_np)
y_test  = torch.from_numpy(y_test_np)

print("X_train:", X_train.shape, " y_train:", y_train.shape)
print("X_test :", X_test.shape,  " y_test :", y_test.shape)
print("입력 차원:", X_train.shape[1])


# 9. 3가지 모델 비교

책과 동일하게 세 가지 모델을 같은 방식으로 학습/평가한다.

| 이름 | 구조                                              | 의미        |
|------|---------------------------------------------------|-------------|
| lr   | `[Linear → Identity]`                             | 선형회귀    |
| nn   | `[Linear → Sigmoid → Linear → Identity]`          | 1 은닉층    |
| dl   | `[Linear → Sigmoid → Linear → Sigmoid → Linear → Identity]` | 2 은닉층 ("deep") |

> 데이터셋이 California Housing 으로 바뀌었기 때문에 절대값 손실은 책과 다르다.
> 핵심은 같은 흐름에서 **층을 깊게 할수록 검증 손실이 더 낮아지는 경향**이 동일하게 재현된다는 점이다.


In [ ]:
INPUT_DIM = X_train.shape[1]
HIDDEN = 13                # 책과 동일한 은닉 너비

torch.manual_seed(SEED)
lr_model = make_network(
    layer_sizes=[INPUT_DIM, 1],
    activations=[nn.Identity()],
)

torch.manual_seed(SEED)
nn_model = make_network(
    layer_sizes=[INPUT_DIM, HIDDEN, 1],
    activations=[nn.Sigmoid(), nn.Identity()],
)

torch.manual_seed(SEED)
dl_model = make_network(
    layer_sizes=[INPUT_DIM, HIDDEN, HIDDEN, 1],
    activations=[nn.Sigmoid(), nn.Sigmoid(), nn.Identity()],
)

print(lr_model)
print(nn_model)
print(dl_model)


In [ ]:
# 모델 1: 선형회귀
trainer = TorchTrainer(lr_model, optim.SGD(lr_model.parameters(), lr=0.01), nn.MSELoss())
trainer.fit(X_train, y_train, X_test, y_test, epochs=50, eval_every=10, seed=SEED)
print()
eval_regression_model(lr_model, X_test, y_test)


In [ ]:
# 모델 2: 은닉층 1개 + sigmoid
trainer = TorchTrainer(nn_model, optim.SGD(nn_model.parameters(), lr=0.01), nn.MSELoss())
trainer.fit(X_train, y_train, X_test, y_test, epochs=50, eval_every=10, seed=SEED)
print()
eval_regression_model(nn_model, X_test, y_test)


In [ ]:
# 모델 3: 은닉층 2개 + sigmoid ("deep")
trainer = TorchTrainer(dl_model, optim.SGD(dl_model.parameters(), lr=0.01), nn.MSELoss())
trainer.fit(X_train, y_train, X_test, y_test, epochs=50, eval_every=10, seed=SEED)
print()
eval_regression_model(dl_model, X_test, y_test)


# 정리: 책의 numpy 코드와 PyTorch 의 매핑

| 책의 추상화 (numpy)              | PyTorch 대응                                |
|----------------------------------|---------------------------------------------|
| `Operation` / `_input_grad`      | `torch.Tensor` + `requires_grad` (autograd) |
| `WeightMultiply` + `BiasAdd`     | `nn.Linear`                                 |
| `Sigmoid`, `Linear`              | `nn.Sigmoid`, `nn.Identity`                 |
| `Layer.forward / backward`       | `nn.Module.forward` (+ autograd backward)   |
| `Dense`                          | `DenseBlock` (Linear + activation)          |
| `Loss.forward / backward`        | `nn.MSELoss` + `loss.backward()`            |
| `NeuralNetwork`                  | `nn.Sequential` / 사용자 `nn.Module`        |
| `SGD.step`                       | `torch.optim.SGD.step` (+ `zero_grad`)      |
| `Trainer.generate_batches`       | `DataLoader(shuffle=True, batch_size=...)`  |
| `Trainer.fit` (early stop)       | `TorchTrainer.fit` (직접 구현)              |

PyTorch 가 자동으로 처리해 주는 부분(autograd, 파라미터 등록, 기울기 누적/초기화)이
책에서 손으로 적었던 모든 backward 식과 `_param_grad` 구현을 대체한다는 것을 확인할 수 있다.
